# Variational Inference for Bingham Mixture Model (BinMM)

This notebook provides a **self-contained and reproducible demonstration** of the proposed method in the paper:

> *Variational Inference for the Bingham Mixture Model and Applications*

The goal of this notebook is to allow readers to:
- understand the core idea of the proposed model,
- reproduce the synthetic experiments presented in the paper,
- verify the behavior of the method in a controlled setting.

---

## Overview

The Bingham distribution is widely used to model **axially symmetric directional data** on the unit hypersphere. However, its practical use is limited by the **intractable normalization constant**, which makes inference difficult.

To address this, the proposed approach:
- reformulates the Bingham mixture model via a **zero-mean Gaussian mixture model (GMM)**,
- performs inference using **variational Bayesian techniques**,
- maps the learned Gaussian parameters back to Bingham parameters via eigendecomposition.

This notebook demonstrates this pipeline step by step.

## 1. Core Implementation

This section defines the core components required for the proposed method, including:

- `VIBinMM`: the main class implementing the variational inference framework,
- conversion from Gaussian mixture parameters to Bingham parameters,
- sampling from the Bingham distribution,
- synthetic data generation utilities,
- clustering evaluation metrics.

The implementation follows the formulation described in the paper, where inference is performed in the Gaussian space and then mapped to the Bingham distribution.

In [7]:
import numpy as np

from sklearn.mixture import BayesianGaussianMixture
from numpy.linalg import inv
from numpy import linalg as LA

from scipy.optimize import linear_sum_assignment as linear_assignment
from sklearn.metrics.cluster import normalized_mutual_info_score as NMI, adjusted_rand_score as ARI

import matplotlib.pyplot as plt
import time

class VIBinMM:

    def __init__(self, n_components=1, max_iter=100, thred=1e-2, tol=1e-3, n_init=1, init_params='random', \
                 mean_precision_prior=100, mean_prior=None, random_state=None):
        '''
        mean_prior: array-like, shape(n_features,), default=[0,...,0]
        '''

        self.D = None
        self.J = n_components

        self.max_iter = max_iter
        self.n_init = n_init
        self.tol = tol
        self.init_params = init_params
        self.mean_precision_prior = mean_precision_prior
        self.mean_prior = mean_prior
        self.random_state = random_state

        self.thred = thred

        self.weight_bhm = None
        self.M_bhm = None
        self.Z_bhm = None

        self.Mus_gau = None
        self.Covs_gau = None

        self.mu = None

        self.gau_model = None

    def init_top_params(self, data):
        self.D = data.shape[1]

        self.weight_bhm = np.ones(self.J) / self.J
        self.M_bhm = np.ones((self.J, self.D, self.D))
        self.Z_bhm = np.ones((self.J, self.D))

        self.mu = np.ones((self.J, self.D))
        self.mu = self.mu / np.linalg.norm(self.mu, axis=1)[:, np.newaxis]

        if self.mean_prior is None:
            self.mean_prior = np.zeros(self.D)

    def GMM2BinMM(self, data):

        bgm = BayesianGaussianMixture(n_components=self.J, tol=self.tol, init_params=self.init_params,
                                      mean_precision_prior=self.mean_precision_prior, \
                                      mean_prior=self.mean_prior, max_iter=self.max_iter, \
                                      n_init=self.n_init).fit(data)

        self.gau_model = bgm

        self.weight_bhm = bgm.weights_

        self.Mus_gau = bgm.means_
        self.Covs_gau = bgm.covariances_

        for i in range(self.J):
            _A = -0.5 * inv(bgm.covariances_[i])
            _Z, _invM = LA.eig(_A)

            self.Z_bhm[i] = _Z
            self.M_bhm[i] = inv(_invM)

            self.mu[i] = self.M_bhm[i, np.argmax(self.Z_bhm[i])]

        # Z tuning
        self.Z_bhm[self.Z_bhm == np.max(self.Z_bhm, axis=1)[:, np.newaxis]] = 0

        # M and Z sorting
        self.M_bhm = np.take_along_axis(self.M_bhm, np.argsort(np.repeat(np.expand_dims(self.Z_bhm, -1), self.Z_bhm.shape[1], axis=-1), axis=1), axis=1)
        self.Z_bhm = np.take_along_axis(self.Z_bhm, np.argsort(self.Z_bhm), axis=1)

    def update_params_byThred(self):

        mask = np.where(self.weight_bhm > self.thred)

        self.weight_bhm = self.weight_bhm[mask]
        self.M_bhm = self.M_bhm[mask]
        self.Z_bhm = self.Z_bhm[mask]

        self.Mus_gau = self.Mus_gau[mask]
        self.Covs_gau = self.Covs_gau[mask]

        self.mu = self.mu[mask]

        self.J = np.size(self.weight_bhm)

    def update_params_byWgtDesc(self):

        sorted_indices = np.argsort(self.weight_bhm)[::-1]

        self.weight_bhm = self.weight_bhm[sorted_indices]
        self.M_bhm = self.M_bhm[sorted_indices]
        self.Z_bhm = self.Z_bhm[sorted_indices]

        self.Mus_gau = self.Mus_gau[sorted_indices]
        self.Covs_gau = self.Covs_gau[sorted_indices]

        self.mu = self.mu[sorted_indices]

    def display_params(self):

        print('Weights: ', self.weight_bhm)
        print('Ms: ', self.M_bhm)
        print('Zs: ', self.Z_bhm)
        print('Mus: ', self.mu)

        print('Mus_gau: ', self.Mus_gau)
        print('Covs_gau: ', self.Covs_gau)

    def fit(self, data, verbose=0, wgtdesc=True):

        self.init_top_params(data)
        self.GMM2BinMM(data)
        self.update_params_byThred()

        if wgtdesc:
            self.update_params_byWgtDesc()

        if verbose:
            self.display_params()

        return self

    def predict_gau_proba(self, data):

        mask = np.where(self.gau_model.weights_ > self.thred)

        pred_prob = self.gau_model.predict_proba(data)
        pred_prob = np.squeeze(pred_prob[:, mask])

        return pred_prob

    def predict(self, data):
        mask = np.where(self.gau_model.weights_ > self.thred)

        pred_prob = self.gau_model.predict_proba(data)
        pred_prob = np.squeeze(pred_prob[:, mask])

        pred = np.argmax(pred_prob, axis=1)

        return pred

    @staticmethod
    def _rv(E, K, n):
        '''
        https://github.com/edur409/Bingham_distributions
        The Bingham distribution from the principal directions and concentrations
        k1 and k2.  k1 and k2 must be negative numbers
        ################################
        #### Simulating from a Bingham distribution based on the principal directions
        #### inferred from a set of poles and their concentrations.
        ################################

        ######### Simulation using any symmetric matrix A
        xc,yc,zc=pbingham(n,E,k1,k2)
        ######### Output
        xc,yc,zc are the coordinates of unit vectors on the sphere
        '''
        from numpy.random import random as runif
        p = len(E) - 1

        lam = -K

        nsamp = 0
        X = []
        qa = len(lam)
        mu = np.zeros(qa)  # Zero means
        sigacginv = 1 + 2 * lam
        SigACG = np.sqrt(1 / (1 + 2 * lam))  # standard deviations
        Ntry = 0

        while nsamp < n:
            xsamp = False
            while (not xsamp):
                yp = np.random.normal(mu, SigACG, qa)
                y = yp / np.sqrt(np.sum(yp ** 2))
                lratio = -np.sum(y ** 2 * lam) - qa / 2 * np.log(qa) + 0.5 * (qa - 1) + qa / 2 * np.log(
                    np.sum(y ** 2 * sigacginv))
                if (np.log(runif(1)) < lratio):
                    X = np.append(X, y)
                    xsamp = True
                    nsamp = nsamp + 1
                Ntry = Ntry + 1
        x = X.reshape((n, qa))  # n normal vectors
        X = E.T @ x.T
        return X.T

    def rvs(self, size):

        size_by_classes = np.round(size * self.weight_bhm)
        size_by_classes[np.argmax(size_by_classes)] += size - np.sum(size_by_classes)
        size_by_classes = size_by_classes.astype(int)

        print('size_by_classes:', size_by_classes)

        assert np.sum(size_by_classes) == size

        labels = np.repeat(np.arange(len(size_by_classes)), size_by_classes)

        samples = []
        for i in range(self.J):
            samples.append(self._rv(self.M_bhm[i], self.Z_bhm[i], size_by_classes[i]))

        samples = np.vstack(samples)

        return samples, labels


def pbingham_EKK(n,E,K):
    '''https://github.com/edur409/Bingham_distributions
    The Bingham distribution from the principal directions and concentrations
    k1 and k2.  k1 and k2 must be negative numbers
    ################################
    #### Simulating from a Bingham distribution based on the principal directions
    #### inferred from a set of poles and their concentrations.
    ####
    ################################

    ######### Simulation using any symmetric matrix A
    xc,yc,zc=pbingham(n,E,k1,k2)
    ######### Output
    xc,yc,zc are the coordinates of unit vectors on the sphere'''
    from numpy.random import random as runif
    p=len(E)-1
    #print('Dimension of Square Matrix',p)
    #lam,E=LA.eig(A)
    #print('The eigenvalues: ',lam)
    #print('The eigenvectors \n: ',V)

    lam=-K

    #lam,E=sort_eig(lam,E)
    #print('The eigenvalues substracting the last term: ',lam)
    #print('The sorted eigenvalues: ',lam)
    #print('The sorted eigenvectors \n: ',V)
    nsamp=0
    X=[]
    qa=len(lam)
    mu=np.zeros(qa) #Zero means
    sigacginv = 1 + 2 * lam
    # SigACG = np.sqrt( 1 / ( 1 + 2 * lam ) ) #standard deviations
    SigACG = np.sqrt(1 / (1 + 2 * lam))  # standard deviations
    Ntry = 0
    #print('mu vector: ',mu)
    #print('Length of eigenvalue vector: ',qa)
    #print('sigacginv :',sigacginv)
    #print('SigACG :',SigACG)
    while nsamp < n:
        xsamp=False
        while (not xsamp):
            yp=np.random.normal(mu,SigACG,qa)
            y=yp/np.sqrt(np.sum(yp**2))
            lratio = -np.sum(y**2 *lam)-qa/2 *np.log(qa)+0.5*(qa-1)+qa/2 *np.log(np.sum(y**2 *sigacginv))
            # lratio = -np.sum(y ** 2 * lam) - qa / 2 * np.log(qa) + (qa - 1) + qa / 2 * np.log(
            #     np.sum(y ** 2 * sigacginv))
            if (np.log(runif(1)) < lratio):
                X=np.append(X,y)
                xsamp=True
                nsamp=nsamp+1
            Ntry=Ntry+1
    x=X.reshape((n,qa)) #n normal vectors
    xc,yc,zc=E.T@x.T
    return xc,yc,zc


def get_synth_config(config_id=1):
    """
    Return one synthetic-data configuration.
    config_id=1: balanced, simpler setting
    config_id=2: unbalanced, mixed shapes
    """

    if config_id == 1:
        config = {
            "n_components": 3,
            "N": [1000, 1000, 1000],
            "D": 3,
            "M": [
                np.array([[0, 0, 1],
                          [0, 1, 0],
                          [1, 0, 0]]),
                np.array([[1, 0, 0],
                          [0, 0, 1],
                          [0, 1, 0]]),
                np.array([[1, 0, 0],
                          [0, 1, 0],
                          [0, 0, 1]])
            ],
            "Z": [
                np.array([-10, -10, 0]),
                np.array([-20, -20, 0]),
                np.array([-30, -30, 0])
            ]
        }

    elif config_id == 2:
        config = {
            "n_components": 3,
            "N": [1000, 2000, 3000],
            "D": 3,
            "M": [
                np.array([[0, 0, 1],
                          [0, 1, 0],
                          [1, 0, 0]]),
                np.array([[1, 0, 0],
                          [0, 0, 1],
                          [0, 1, 0]]),
                np.array([[1, 0, 0],
                          [0, 1, 0],
                          [0, 0, 1]])
            ],
            "Z": [
                np.array([-40, 0, 0]),
                np.array([-40, -20, 0]),
                np.array([-40, -40, 0])
            ]
        }

    else:
        raise ValueError(f"Unknown config_id: {config_id}")

    return config


def generate_synthetic_dataset(config, seed=None):
    """
    Generate one independent synthetic dataset from a given config.
    Returns:
        X      : (N_total, D)
        labels : (N_total,)
        params : dict of ground-truth parameters
    """
    if seed is not None:
        np.random.seed(seed)

    n_components = config["n_components"]
    N = config["N"]
    D = config["D"]
    M = config["M"]
    Z = config["Z"]

    X_list = []
    labels = []

    for k in range(n_components):
        samples = pbingham_EKK(N[k], M[k], Z[k])
        X_k = np.column_stack(samples)

        X_list.append(X_k)
        labels.append(np.full(N[k], k, dtype=int))

    X = np.vstack(X_list)
    labels = np.concatenate(labels)

    params = {
        "n_components": n_components,
        "N": N,
        "D": D,
        "M": M,
        "Z": Z
    }

    return X, labels, params

def cluster_acc(Y_pred, Y):
    assert Y_pred.size == Y.size
    D = max(Y_pred.max(), Y.max()) + 1
    w = np.zeros((D, D), dtype=np.int64)
    for i in range(Y_pred.size):
        w[Y_pred[i], Y[i]] += 1
    ind = linear_assignment(w.max() - w)
    total = 0
    for i in range(len(ind[0])):
        total += w[ind[0][i], ind[1][i]]
    return total * 1.0 / Y_pred.size, w

## 2. Experimental Pipeline

This section implements the full experimental pipeline used in the synthetic studies.

Each experiment consists of the following steps:
1. Generate a synthetic dataset from known Bingham distributions,
2. Fit the proposed VIBinMM model,
3. Predict cluster assignments,
4. Evaluate clustering performance using standard metrics.

To ensure robustness, we repeat the experiment multiple times with different random seeds and report the average performance.

In [5]:
def run_single_experiment(config_id=2, seed=0, max_iter=1500, thred=4e-2, init_params='random_from_data', verbose=0):
    """
    Run one independent experiment:
      1) generate data
      2) fit model
      3) evaluate cluster_acc / ARI / NMI
    """
    config = get_synth_config(config_id=config_id)
    X, y_true, params = generate_synthetic_dataset(config, seed=seed)

    model = VIBinMM(
        n_components=config["n_components"],
        max_iter=max_iter,
        thred=thred,
        init_params=init_params
    )

    start = time.time()
    model.fit(X, verbose=verbose)
    runtime = time.time() - start

    y_pred = model.predict(X)

    acc = cluster_acc(y_true, y_pred)[0]
    ari = ARI(y_true, y_pred)
    nmi = NMI(y_true, y_pred)

    result = {
        "seed": seed,
        "acc": acc,
        "ari": ari,
        "nmi": nmi,
        "runtime": runtime
    }

    return result


def run_simulation(config_id=2, S=10, max_iter=1500, thred=4e-2, init_params='random_from_data', seed=1, verbose=0):
    """
    Run S independent experiments under the same configuration.
    Seeds are 0, 1, ..., S-1.
    Report mean/std of cluster_acc, ARI, and NMI.
    """
    acc_list = []
    ari_list = []
    nmi_list = []
    runtime_list = []

    print(f"\n========== Simulation starts: config_id = {config_id}, S = {S} ==========")

    for i in range(S):

        if seed:
          print(f"\n----- Run {i+1}/{S} | seed = {i} -----")

          res = run_single_experiment(
              config_id=config_id,
              seed=i,
              max_iter=max_iter,
              thred=thred,
              init_params=init_params,
              verbose=verbose
          )
        else:
          print(f"\n----- Run {i+1}/{S} -----")

          res = run_single_experiment(
              config_id=config_id,
              seed=None,
              max_iter=max_iter,
              thred=thred,
              verbose=verbose
          )


        acc_list.append(res["acc"])
        ari_list.append(res["ari"])
        nmi_list.append(res["nmi"])
        runtime_list.append(res["runtime"])

        print("cluster_acc = {:.6f}".format(res["acc"]))
        print("ARI         = {:.6f}".format(res["ari"]))
        print("NMI         = {:.6f}".format(res["nmi"]))
        print("runtime     = {:.6f} s".format(res["runtime"]))

    results = {
        "config_id": config_id,
        "S": S,
        "acc_all": acc_list,
        "ari_all": ari_list,
        "nmi_all": nmi_list,
        "runtime_all": runtime_list,
        "acc_mean": float(np.mean(acc_list)),
        "acc_std": float(np.std(acc_list)),
        "ari_mean": float(np.mean(ari_list)),
        "ari_std": float(np.std(ari_list)),
        "nmi_mean": float(np.mean(nmi_list)),
        "nmi_std": float(np.std(nmi_list)),
        "runtime_mean": float(np.mean(runtime_list)),
        "runtime_std": float(np.std(runtime_list)),
    }

    print("\n========== Final Results ==========")
    print("Config ID   : {}".format(results["config_id"]))
    print("S           : {}".format(results["S"]))
    print("cluster_acc : {:.6f} ± {:.6f}".format(results["acc_mean"], results["acc_std"]))
    print("ARI         : {:.6f} ± {:.6f}".format(results["ari_mean"], results["ari_std"]))
    print("NMI         : {:.6f} ± {:.6f}".format(results["nmi_mean"], results["nmi_std"]))
    print("runtime     : {:.6f} ± {:.6f} s".format(results["runtime_mean"], results["runtime_std"]))

    return results

## 3. Running the Experiment

We now execute the full experimental pipeline using the previously defined functions.

In this example, we use **Configuration 1**, which corresponds to a balanced synthetic dataset with three Bingham components of similar structure.

The simulation is repeated multiple times to ensure robustness and to obtain statistically meaningful results.

In [8]:
results = run_simulation(config_id=1, S=10, max_iter=1500, thred=4e-2, init_params='random_from_data', verbose=0)


========== Simulation starts: config_id = 1, S = 10 ==========

----- Run 1/10 | seed = 0 -----
cluster_acc = 0.998667
ARI         = 0.996005
NMI         = 0.991245
runtime     = 0.293354 s

----- Run 2/10 | seed = 1 -----
cluster_acc = 0.999667
ARI         = 0.999000
NMI         = 0.997601
runtime     = 0.211663 s

----- Run 3/10 | seed = 2 -----
cluster_acc = 0.999000
ARI         = 0.997002
NMI         = 0.993223
runtime     = 0.263726 s

----- Run 4/10 | seed = 3 -----
cluster_acc = 0.999667
ARI         = 0.999000
NMI         = 0.997601
runtime     = 0.232048 s

----- Run 5/10 | seed = 4 -----
cluster_acc = 0.999667
ARI         = 0.999000
NMI         = 0.997601
runtime     = 0.133555 s

----- Run 6/10 | seed = 5 -----
cluster_acc = 0.999000
ARI         = 0.997002
NMI         = 0.993223
runtime     = 0.215311 s

----- Run 7/10 | seed = 6 -----
cluster_acc = 0.998667
ARI         = 0.996004
NMI         = 0.990824
runtime     = 0.037729 s

----- Run 8/10 | seed = 7 -----
cluster_acc = 